# WarehousePG Backup and Restore

## Init
Create a database schema and a single table to be used for backup and recovery.

In [ ]:
# variables for demo
bucket_name = "warehousepg-backups"
region = "us-east-1"
folder = "whpg-demo"

from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

!pip install boto3

#Cleanup
!aws s3 rm --recursive s3://{bucket_name}/{folder}

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

## Demo Table
Create a schema and table with 100,000 rows of data that will be used for this demo.

In [ ]:
%%sql
DROP SCHEMA IF EXISTS foo CASCADE;
CREATE SCHEMA foo;

CREATE TABLE foo.bar AS SELECT i FROM generate_series(1, 100000) as i DISTRIBUTED BY (i);

### Query
Show the data in the table.

In [ ]:
%%sql

SELECT *
FROM foo.bar;

## `gpbackup`
This utility takes a backup by dumping all of the database files in parallel. The backup destination for this demo is s3. The following is an example s3 configuration file for gpbackup.

### Create Configuration file
Create a yaml file for the `gpbackup` demo.

In [ ]:
config = f"""
executablepath: $GPHOME/bin/gpbackup_s3_plugin
options:
  region: {region}
  bucket: {bucket_name}
  folder: {folder}
  encryption: "on"
"""

with open("gpbackup_demo.yaml", "w") as f:
    f.write(config)

print(open("gpbackup_demo.yaml").read())
    

### Execute `gpbackup`
Take a full backup of the database. The backup will use the plugin-config file which will put the backup files into s3.

In [ ]:
import os
dbname = os.environ["PGDATABASE"]

!gpbackup --dbname {dbname} --plugin-config /home/gpadmin/gpbackup_demo.yaml

### Drop the `foo` schema and the `bar` table
Ooops! You accidently drop the foo schema which has your bar table you need! You can quickly restore this by using `gprestore`.

In [ ]:
%%sql

DROP SCHEMA foo CASCADE;

### ERROR!
The table is gone! To recover, you don't need to recover the entire database. You can selectively restore just the schema or even just the table. For this demo, we will restore the schema which includes the table.

In [ ]:
%%sql

SELECT *
FROM foo.bar;

### Latest backup
Use python to find the latest backup in S3.

In [ ]:
import re
import boto3

s3 = boto3.client("s3")
prefix = f"{folder}/backups/"
paginator = s3.get_paginator("list_objects_v2")

timestamps = set()
pattern = re.compile(r"(\d{14})")

for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
    for obj in page.get("Contents", []):
        match = pattern.search(obj["Key"])
        if match:
            timestamps.add(match.group(1))

latest = max(timestamps)
print(latest)

### Restore the schema and table with `gprestore`
One of the benefits of `gpbackup` and `gprestore` is the ability to restore specific objects like schemas and tables.

In [ ]:
!gprestore --plugin-config /home/gpadmin/gpbackup_demo.yaml --include-schema foo --timestamp {latest}

### ERROR FREE!
Now that the schema and table have been restored, you can now query the data again.

In [ ]:
%%sql

SELECT *
FROM foo.bar;

## Close the connection

In [ ]:
connection_url = f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}"
%sql --close {{connection_url}}